In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold

In [3]:
df = pd.read_csv("best_data.csv")

In [4]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [5]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [6]:
# Impute
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)


## Mean Baseline

ใช้ค่าเฉลี่ยของ `Average_Bleaching` จาก train set เป็นคำตอบทุกแถว เพื่อเป็น naive baseline ว่าโมเดลของเราทำ RMSE ได้ดีกว่าการเดาค่าเฉลี่ยหรือไม่


In [7]:
rmse = 7.6157
r2 = 0.3672

In [8]:
# Mean baseline: predict every test row with train target mean
mean_baseline_value = y_train.mean()
mean_baseline_preds = np.full_like(y_test, mean_baseline_value, dtype=float)

mean_baseline_r2 = r2_score(y_test, mean_baseline_preds)
mean_baseline_rmse = np.sqrt(mean_squared_error(y_test, mean_baseline_preds))

print("========== MEAN BASELINE (80/20 split) ==========")
print(f"Mean Average_Bleaching from train set: {mean_baseline_value:.4f}")
print(f"Mean baseline R²:   {mean_baseline_r2:.4f}")
print(f"Mean baseline RMSE: {mean_baseline_rmse:.4f}")

if 'rmse' in globals() and 'r2' in globals():
    print()
    print("========== COMPARE WITH OUR ENSEMBLE ==========")
    print(f"Our ensemble R²:   {r2:.4f}")
    print(f"Our ensemble RMSE: {rmse:.4f}")
    print(f"RMSE improvement over mean baseline: {mean_baseline_rmse - rmse:.4f}")

    if rmse < mean_baseline_rmse:
        print("✅ Our ensemble RMSE is better than mean baseline.")
    else:
        print("❌ Our ensemble RMSE is not better than mean baseline.")
else:
    print("Run the ensemble evaluation cell first to compare with our model.")


========== MEAN BASELINE (80/20 split) ==========
Mean Average_Bleaching from train set: 2.7706
Mean baseline R²:   -0.0002
Mean baseline RMSE: 9.6871

========== COMPARE WITH OUR ENSEMBLE ==========
Our ensemble R²:   0.3672
Our ensemble RMSE: 7.6157
RMSE improvement over mean baseline: 2.0714
✅ Our ensemble RMSE is better than mean baseline.


In [9]:
# K-Fold Mean Baseline: each fold predicts with that fold's train target mean
n_splits = 4
kf_mean_baseline = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mean_baseline_kfold_results = []

for fold, (train_idx, test_idx) in enumerate(kf_mean_baseline.split(X), 1):
    y_fold_train = Y.iloc[train_idx]
    y_fold_test = Y.iloc[test_idx]

    fold_mean = y_fold_train.mean()
    fold_preds = np.full_like(y_fold_test, fold_mean, dtype=float)

    fold_r2 = r2_score(y_fold_test, fold_preds)
    fold_rmse = np.sqrt(mean_squared_error(y_fold_test, fold_preds))

    mean_baseline_kfold_results.append({
        'Fold': fold,
        'Train_Mean_Average_Bleaching': fold_mean,
        'R²': fold_r2,
        'RMSE': fold_rmse,
    })

    print(f"Fold {fold} → mean: {fold_mean:.4f} | R²: {fold_r2:.4f} | RMSE: {fold_rmse:.4f}")

mean_baseline_kfold_df = pd.DataFrame(mean_baseline_kfold_results)

print()
print(f"--- {n_splits}-Fold Mean Baseline Summary ---")
print(f"R²   mean: {mean_baseline_kfold_df['R²'].mean():.4f} ± {mean_baseline_kfold_df['R²'].std():.4f}")
print(f"RMSE mean: {mean_baseline_kfold_df['RMSE'].mean():.4f} ± {mean_baseline_kfold_df['RMSE'].std():.4f}")

display(mean_baseline_kfold_df)


Fold 1 → mean: 2.8285 | R²: -0.0002 | RMSE: 9.3734
Fold 2 → mean: 2.6633 | R²: -0.0024 | RMSE: 10.8880
Fold 3 → mean: 2.7309 | R²: -0.0007 | RMSE: 9.7413
Fold 4 → mean: 2.9582 | R²: -0.0061 | RMSE: 8.3813

--- 4-Fold Mean Baseline Summary ---
R²   mean: -0.0023 ± 0.0027
RMSE mean: 9.5960 ± 1.0353


,Fold,Train_Mean_Average_Bleaching,R²,RMSE
0,1,2.828460,-0.000201,9.373428
1,2,2.663343,-0.002353,10.887962
2,3,2.730890,-0.000698,9.741314
3,4,2.958185,-0.006086,8.381296
